# Build a Kraken adapter

Define the protocol in Python, connect a source, and serve its books in the terminal. All of the protocol definition is below.

Start with the short recorded-input example, then connect the real source and open the terminal. Price and quantity outputs use integer units at the declared precisions.

In [1]:
from pathlib import Path
from time import perf_counter
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "rust/crates/lobo_replay").is_dir())
sys.path.insert(0, str(root / "notebooks"))
from adapter_examples import wait_for_book
import pandas as pd
from IPython.display import display, HTML
from lobo.server import server_context

## Define the protocol

These declarations are the adapter definition. Edit the layouts, conditions or mappings here to change how your source drives the books.

In [2]:
from lobo.replay.adapters import CustomAdapter, Protocol
from lobo.replay.adapters import expressions as le
from lobo.replay.adapters import models as lm


def protocol(depth: int = 100) -> Protocol:
    if depth not in (10, 25, 100, 500, 1000):
        raise ValueError("depth must be 10, 25, 100, 500, or 1000")
    symbol = le.Variable("symbol")
    return Protocol(
        lm.Json(
            lm.Message(le.Field("success").eq(False), lm.Fail("Subscription request failed")),
            lm.Message(
                le.Field("channel").eq("instrument"),
                le.ForEach(
                    le.Field("data", "pairs"),
                    le.When(
                        le.Field("status").ne("delisted")
                        & le.Field("status").ne("work_in_progress"),
                        lm.Register(
                            symbol=le.Field("symbol"),
                            price_decimals=le.Field("price_precision"),
                            quantity_decimals=le.Field("qty_precision"),
                        ),
                    ),
                ),
                lm.Subscribe(),
            ),
            lm.Message(
                le.Field("channel").eq("book"),
                le.ForEach(
                    le.Field("data"),
                    lm.Book(
                        le.Field("symbol"),
                        le.ForEach(
                            le.Field("asks"),
                            lm.Level(
                                side="sell",
                                price=le.Field("price").decimal(
                                    le.Variable("price_decimals")
                                ),
                                quantity=le.Field("qty").decimal(
                                    le.Variable("quantity_decimals")
                                ),
                            ),
                        ),
                        le.ForEach(
                            le.Field("bids"),
                            lm.Level(
                                side="buy",
                                price=le.Field("price").decimal(
                                    le.Variable("price_decimals")
                                ),
                                quantity=le.Field("qty").decimal(
                                    le.Variable("quantity_decimals")
                                ),
                            ),
                        ),
                        snapshot=le.Root("type").eq("snapshot"),
                        timestamp=le.Field("timestamp").timestamp("rfc3339"),
                        depth=depth,
                        checksum=lm.Checksum(
                            le.Field("checksum"),
                            depth=10,
                            on_failure=(
                                lm.Send(
                                    {
                                        "method": "unsubscribe",
                                        "params": {
                                            "channel": "book",
                                            "symbol": [symbol],
                                            "depth": depth,
                                        },
                                    }
                                ),
                                lm.Send(
                                    {
                                        "method": "subscribe",
                                        "params": {
                                            "channel": "book",
                                            "symbol": [symbol],
                                            "depth": depth,
                                            "snapshot": True,
                                        },
                                    }
                                ),
                            ),
                        ),
                    ),
                ),
            ),
            lm.Message(
                le.Field("channel").eq("trade") & le.Field("type").eq("update"),
                le.ForEach(
                    le.Field("data"),
                    lm.Book(
                        le.Field("symbol"),
                        lm.Trade(
                            id=le.Field("trade_id"),
                            price=le.Field("price").decimal(le.Variable("price_decimals")),
                            quantity=le.Field("qty").decimal(
                                le.Variable("quantity_decimals")
                            ),
                            maker_side=le.Field("side").map(
                                {"buy": "sell", "sell": "buy"}
                            ),
                            timestamp=le.Field("timestamp").timestamp("rfc3339"),
                        ),
                        timestamp=le.Field("timestamp").timestamp("rfc3339"),
                        ready=False,
                    ),
                ),
            ),
        ),
        connect=(
            lm.Send(
                {
                    "method": "subscribe",
                    "params": {"channel": "instrument", "snapshot": True},
                }
            ),
        ),
        keepalive=(lm.Send({"method": "ping"}),),
        subscriptions=(
            lm.Send(
                {
                    "method": "subscribe",
                    "params": {
                        "channel": "book",
                        "symbol": [symbol],
                        "depth": depth,
                        "snapshot": True,
                    },
                }
            ),
            lm.Send(
                {
                    "method": "subscribe",
                    "params": {
                        "channel": "trade",
                        "symbol": [symbol],
                        "snapshot": False,
                    },
                }
            ),
        ),
    )

## Construct, then consume input

`CustomAdapter(protocol(), source, ...)` validates and compiles the declaration with Cranelift during construction. There is no separate compile call. The source then supplies raw bytes to the compiled packet program. Changing a declaration creates a different program; identical definitions can reuse the process-local cache.

The timings below separate construction from `start()` → `wait()`. Construction includes declaration conversion and preparation; it is **not a compiler-only measurement**. Starting also includes worker setup and, when serving, preparation for publishing to observers. These tiny examples demonstrate behavior, not throughput.

JSON uses the compiled Rust parser to produce a value tree, followed by generated field extraction and control flow. It still allocates that tree. Definition validation is done at construction; incoming syntax, sequence and checksum handling remain part of consuming the feed.

In [3]:
fixtures = root / "rust/crates/lobo_adapters/tests/fixtures"
# A recorded directory establishes the integer price/quantity scales.
directory = b'{"channel":"instrument","type":"snapshot","data":{"pairs":[{"symbol":"BTC/USD","price_precision":1,"qty_precision":8,"status":"online"}]}}'
source = lm.Source.packets([
    directory,
    (fixtures / "kraken_snapshot.json").read_bytes(),
    (fixtures / "kraken_update.json").read_bytes(),
])
definition = protocol(depth=10)
started = perf_counter()
fixture_adapter = CustomAdapter(
    definition, source, name="Kraken recording", symbol="BTC/USD", scope=["BTC/USD"],
    mode="live", level="l2",
)
construction_ms = (perf_counter() - started) * 1000

with fixture_adapter:
    started = perf_counter()
    fixture_adapter.start()
    fixture_adapter.wait()
    completion_ms = (perf_counter() - started) * 1000
    status = fixture_adapter.status()
    levels = pd.DataFrame(fixture_adapter.levels("BTC/USD"))
    assert status["checksum_checks"] == 2 and status["checksum_failures"] == 0
    assert levels.loc[levels["price"] == "452835", "quantity"].iloc[0] == 20_000_000
    display(levels.head(12))
    fixture_adapter.simulate("buy", 1_000_000)
    display(pd.DataFrame(fixture_adapter.simulation_report()["executions"]))

print(f"Construction: {construction_ms:.3f} ms | Start to completion: {completion_ms:.3f} ms")
print(f"Consumed {status['bytes']:,} bytes / {status['messages']} messages")
print(f"Checksums: {status['checksum_checks']} passed, {status['checksum_failures']} failed")

,hidden,orders,price,quantity,side
0,0,0,452840,30000000,buy
1,0,0,452835,20000000,buy
2,0,0,452834,154582015,buy
3,0,0,452821,10000000,buy
4,0,0,452810,10000000,buy
5,0,0,452803,154592586,buy
6,0,0,452790,7990000,buy
7,0,0,452776,3310103,buy
8,0,0,452775,30000000,buy
9,0,0,452773,154602737,buy


,price,quantity,sequence,simulated,timestamp_ns
0,452852,100000,1,True,1788703200123456000
1,452864,900000,2,True,1788703200123456000


Construction: 3.615 ms | Start to completion: 0.128 ms
Consumed 2,442 bytes / 3 messages
Checksums: 2 passed, 0 failed


## Connect the source and serve its books

The same declaration drives the public WebSocket. The scope fixes which books the session runs. This section requires internet access; no credentials are needed.

In [4]:
symbol = 'BTC/USD'
definition = protocol()
started = perf_counter()
adapter = CustomAdapter(
    definition, lm.Source.websocket("wss://ws.kraken.com/v2"),
    name='Kraken', symbol=symbol, scope=[symbol], mode='live', level='l2',
    timezone='UTC',
)
print(f"Construction (may reuse compiled code): {(perf_counter() - started) * 1000:.3f} ms")

Construction (may reuse compiled code): 2.480 ms


Open the link to see the charts and use the simulation controls. The server stays running until the cleanup cell.

Run the cells individually to keep the terminal open while you explore. Run All reaches cleanup and closes it.

In [5]:
terminal = server_context(adapters=[adapter], port=0)
display(HTML(f'<a href="{terminal.url}" target="_blank">Open order-book terminal</a>'))

In [6]:
levels = wait_for_book(adapter, symbol)
display(pd.DataFrame(levels).head(12))
adapter.status()

,hidden,orders,price,quantity,side
0,0,0,789009,47528,buy
1,0,0,788995,5100,buy
2,0,0,788972,7041514,buy
3,0,0,788971,63373633,buy
4,0,0,788966,30177069,buy
5,0,0,788965,6924559,buy
6,0,0,788956,5100,buy
7,0,0,788949,6945,buy
8,0,0,788945,95045937,buy
9,0,0,788931,20502355,buy


{'books': 1,
 'bytes': 578535,
 'checksum_checks': 2,
 'checksum_failures': 0,
 'clock_ns': 1788932741200246959,
 'complete': False,
 'messages': 7,
 'symbol': 'BTC/USD',
 'synchronized': True}

Preview a market order and inspect its execution report.

In [7]:
adapter.simulate("buy", 1000000)
adapter.simulation_report()

{'alternate_timeline': False,
 'average_price': 789010.0,
 'complete': True,
 'executions': [{'price': 789010,
   'quantity': 1000000,
   'sequence': 1,
   'simulated': True,
   'timestamp_ns': 1788932741210531250}],
 'filled': 1000000,
 'ignored': 0,
 'order_id': '10f8090d-2897-4a1c-9921-4aa0676278ad',
 'remaining': 0,
 'requested': 1000000,
 'simulated': True,
 'stopped': True,
 'symbol': 'BTC/USD'}

Run this cell when finished exploring the terminal.

In [8]:
terminal.close()

## Recorded performance results

The September 9, 2026 paired benchmark for **100 snapshots** measured **1,112.8 µs** for the existing adapter and **1,344.4 µs** for the compiled custom adapter, a **1.21×** paired ratio. Streaming parity has not been reached. These are saved benchmark results, not timings from this notebook. The benchmark excludes construction and compilation; the demonstration above includes worker startup.

See [validation and reproduction commands](../python/examples/VALIDATION.md) for all four feeds and the timing boundaries.